## Methodology note & data-caveats

- The baseline metrics shown here are produced by `src/models/baseline_forecasting.py` and stored in `baseline_results.csv`.
- **MA-28 is the selected primary forecasting method.** Per the Milestone-11 evaluation audit (`src/models/evaluation_audit.py` and `data/processed/audited_model_comparison.csv`), an earlier ML-vs-baseline comparison used an **invalid frozen MA-28**: a look-ahead constant equal to the mean of the last 28 days of the whole dataset, applied unchanged to every test row. That value leaked future test-period demand and is **not a valid per-date forecast**; its historical `MA-28 Test MAE = 1.1501` is invalid and no longer authoritative.
- The authoritative, leakage-safe, apples-to-apples comparison is the **common evaluation population** recomputed by the audit: MA-28 validation MAE **1.0820** / test MAE **1.1138** (Random Forest 1.0860/1.1120, XGBoost 1.0897/1.1136). MA-28 wins on validation; the ML challengers' nominal test edge (+0.17%) is not a basis for selection.
- Aggregation note: the MAE values below are the mean of per-series MAE (macro); the audit reports pooled MAE (micro) on the common population. The audit values above are authoritative.


# Baseline Forecasting Results

## M5 Demand Forecasting - Baseline Model Comparison

**Notebook:** `02_baseline_forecasting.ipynb`

**Date:** 2026-08-31


## 1. Business Objective

This notebook presents the results of baseline forecasting models for the M5 retail demand dataset.
Baseline models establish performance benchmarks that more advanced ML models must beat.

**Objective:** Establish strong forecasting baselines using simple, interpretable models:
- Naive (random walk): forecast(t+1) = demand(t)
- Moving Average 7: forecast(t+1) = mean(demand[t-6:t])
- Moving Average 28: forecast(t+1) = mean(demand[t-27:t])
- Seasonal Naive 7: forecast(t+1) = demand(t-7)


## Setup

Import libraries and load results.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent
RESULTS_PATH = PROJECT_ROOT / 'data' / 'processed' / 'baseline_results.csv'
PREDICTIONS_PATH = PROJECT_ROOT / 'data' / 'processed' / 'baseline_predictions.parquet'

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)

print('Paths configured.')


## 2. Load Results

Load baseline evaluation results and predictions.


In [ ]:
results_df = pd.read_csv(RESULTS_PATH)
predictions_df = pd.read_parquet(PREDICTIONS_PATH)

print('Results shape:', results_df.shape)
print('Predictions shape:', predictions_df.shape)
print()
print('Results columns:', results_df.columns.tolist())
print('Predictions columns:', predictions_df.columns.tolist())


## 3. Train/Validation/Test Split

Visualize the chronological data split.


In [ ]:
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'demand_dev.parquet'
df = pd.read_parquet(DATA_PATH)

unique_dates = df['date'].unique()
unique_dates.sort()
n_dates = len(unique_dates)
n_train = int(n_dates * 0.80)
n_val = int(n_dates * 0.10)

train_end = unique_dates[n_train - 1]
val_end = unique_dates[n_train + n_val - 1]

print('Data split summary:')
print(f'  Full dataset: {unique_dates[0]} to {unique_dates[-1]}')
print(f'  Training:    {unique_dates[0]} to {train_end} ({n_train} days)')
print(f'  Validation:  {unique_dates[n_train]} to {val_end} ({n_val} days)')
print(f'  Test:        {unique_dates[n_train + n_val]} to {unique_dates[-1]} ({n_dates - n_train - n_val} days)')

fig, ax = plt.subplots(figsize=(14, 4))
ax.axvspan(unique_dates[0], train_end, alpha=0.3, label='Training (80%)', color='blue')
ax.axvspan(unique_dates[n_train], val_end, alpha=0.3, label='Validation (10%)', color='orange')
ax.axvspan(unique_dates[n_train + n_val], unique_dates[-1], alpha=0.3, label='Test (10%)', color='green')
ax.set_title('Chronological Train/Validation/Test Split', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()


## 4. Model Performance Comparison

Compare baseline models across evaluation metrics.


In [ ]:
summary = results_df.groupby(['split', 'model'])[['mae', 'rmse', 'wape']].mean().round(4)
print('Overall Performance Summary:')
print('=' * 60)
print(summary)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

metrics = ['mae', 'rmse', 'wape']
titles = ['Mean Absolute Error (MAE)', 'Root Mean Squared Error (RMSE)', 'Weighted Absolute Percentage Error (WAPE)']

for idx, (metric, title) in enumerate(zip(metrics, titles)):
    ax = axes[idx]
    pivot = results_df.groupby('model')[metric].mean().sort_values()
    bars = ax.bar(range(len(pivot)), pivot.values, edgecolor='black')
    ax.set_xticks(range(len(pivot)))
    ax.set_xticklabels(pivot.index, rotation=45, ha='right')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(metric.upper())
    for bar, val in zip(bars, pivot.values):
        ax.annotate(f'{val:.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 3), textcoords='offset points', ha='center', va='bottom')

plt.tight_layout()
plt.show()


## 5. Representative Series Analysis

Visualize actual vs predicted for representative product/store combinations.


In [ ]:
stats = predictions_df.groupby('id')['actual_demand'].agg(['mean', 'std']).reset_index()
stats.columns = ['id', 'mean_demand', 'std_demand']
stats['zero_pct'] = predictions_df.groupby('id')['actual_demand'].apply(lambda x: (x == 0).mean()).values
stats['cv'] = np.where(stats['mean_demand'] > 0, stats['std_demand'] / stats['mean_demand'], np.nan)
high_vol = stats.loc[stats['mean_demand'].idxmax(), 'id']
stable = stats.loc[stats['cv'].idxmin(), 'id']
volatile = stats.loc[stats['cv'].idxmax(), 'id']
intermittent = stats.loc[stats['zero_pct'].idxmax(), 'id']
representatives = {'High-Volume': high_vol, 'Stable': stable, 'Volatile': volatile, 'Intermittent': intermittent}
print('Representative series:')
for name, pid in representatives.items():
    print(f'  {name}: {pid}')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()
for idx, (name, pid) in enumerate(representatives.items()):
    ax = axes[idx]
    data = predictions_df[predictions_df['id'] == pid].sort_values('date')
    ax.plot(data['date'], data['actual_demand'], label='Actual', linewidth=1.5, alpha=0.8)
    ax.plot(data['date'], data['ma_28_prediction'], label='MA-28 (Best)', linewidth=1, alpha=0.7)
    ax.plot(data['date'], data['naive_prediction'], label='Naive', linewidth=1, alpha=0.7, linestyle='--')
    ax.set_title(f'{name}: {pid}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Demand')
    ax.legend(fontsize=9)
    ax.tick_params(axis='x', rotation=45)
plt.suptitle('Actual vs Predicted Demand', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 6. Error Distribution

Analyze the distribution of forecast errors.


In [ ]:
predictions_df['naive_error'] = predictions_df['actual_demand'] - predictions_df['naive_prediction']
predictions_df['ma28_error'] = predictions_df['actual_demand'] - predictions_df['ma_28_prediction']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(predictions_df['naive_error'].dropna(), bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Naive Forecast Error Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Error (Actual - Predicted)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].hist(predictions_df['ma28_error'].dropna(), bins=50, edgecolor='black', alpha=0.7)
axes[1].set_title('MA-28 Forecast Error Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Error (Actual - Predicted)')
axes[1].set_ylabel('Frequency')
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
plt.tight_layout()
plt.show()


## 7. Performance by Demand Segment

Analyze how baseline performance varies by demand type.


In [ ]:
stats = predictions_df.groupby('id')['actual_demand'].agg(['mean']).reset_index()
stats.columns = ['id', 'mean_demand']
stats['zero_pct'] = predictions_df.groupby('id')['actual_demand'].apply(lambda x: (x == 0).mean()).values
median_demand = stats['mean_demand'].median()
def classify(row):
    if row['zero_pct'] > 0.5: return 'Intermittent'
    elif row['mean_demand'] > median_demand: return 'High-Volume'
    else: return 'Low-Volume'
stats['segment'] = stats.apply(classify, axis=1)
pred_with_seg = predictions_df.merge(stats[['id', 'segment']], on='id')
pred_with_seg['naive_mae'] = np.abs(pred_with_seg['actual_demand'] - pred_with_seg['naive_prediction'])
pred_with_seg['ma28_mae'] = np.abs(pred_with_seg['actual_demand'] - pred_with_seg['ma_28_prediction'])
segment_perf = pred_with_seg.groupby('segment')[['naive_mae', 'ma28_mae']].mean().round(4)
print('Performance by Segment (MAE):')
print(segment_perf)
fig, ax = plt.subplots(figsize=(10, 6))
segment_perf.plot(kind='bar', ax=ax, edgecolor='black')
ax.set_title('MAE by Demand Segment', fontsize=14, fontweight='bold')
ax.set_xlabel('Segment')
ax.set_ylabel('Mean Absolute Error')
ax.tick_params(axis='x', rotation=0)
ax.legend(['Naive', 'MA-28'])
plt.tight_layout()
plt.show()


## 8. Baseline Conclusion

### Key Findings

1. **Best Baseline**: The 28-day moving average (MA-28) achieved the lowest MAE and RMSE.
2. **Primary Metric**: MAE is used as the primary metric (robust to zero-demand).
3. **Seasonal Naive**: Did not significantly outperform simple naive.
4. **Moving Averages Help**: Both MA-7 and MA-28 outperformed naive baselines.
5. **Intermittent Demand**: 66.6% zero-demand observations make forecasting challenging.
6. **Limitations**: Baselines do not capture trends, seasonality, or external factors.

### Next Steps

- Feature engineering for trend, seasonality, and price effects
- ML models (Random Forest, XGBoost, LSTM) to beat baselines
- Probabilistic forecasting for inventory optimization
